In [1]:
!pip install mediapipe opencv-python numpy tqdm


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install opencv-python numpy scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip uninstall mediapipe -y
!pip install mediapipe==0.10.14

Found existing installation: mediapipe 0.10.8
Uninstalling mediapipe-0.10.8:
  Successfully uninstalled mediapipe-0.10.8
  Using cached ml_dtypes-0.5.4-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
    --------------------------------------- 0.8/50.8 MB 4.2 MB/s eta 0:00:12
   - -------------------------------------- 2.4/50.8 MB 6.4 MB/s eta 0:00:08
   --- ------------------------------------ 3.9/50.8 MB 6.9 MB/s eta 0:00:07
   --- ------------------------------------ 4.7/50.8 MB 5.8 MB/s eta 0:00:08
   --- ------------------------------------ 5.0/50.8 MB 5.1 MB/s eta 0:00:09
   --- ------------------------------------ 5.0/50.8 MB 5.1 MB/s eta 0:00:09
   ---- ----------------------------------- 5.8/50.8 MB 4.0 MB/s eta 0:00:12
   ----- ---------------------------------- 6.6/50.8 MB 4.0 MB/s eta 0:00:12
   ----- ---------------------------------- 6.8/50.8 MB 4.0 MB/s eta 0:00:12
   ----- --------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.52.2 requires pandas<3,>=1.4.0, which is not installed.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import mediapipe
print(mediapipe)
print(mediapipe.__file__)

<module 'mediapipe' from 'c:\\Users\\Rishu\\AppData\\Local\\Programs\\Python\\Python310\\lib\\site-packages\\mediapipe\\__init__.py'>
c:\Users\Rishu\AppData\Local\Programs\Python\Python310\lib\site-packages\mediapipe\__init__.py


In [1]:
import mediapipe as mp

print("Mediapipe version:", mp.__version__)

# For MediaPipe 0.10.32+, use the tasks API
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("✓ MediaPipe imported successfully")
print("Available vision tasks:", dir(vision))

Mediapipe version: 0.10.32
✓ MediaPipe imported successfully
Available vision tasks: ['FaceDetector', 'FaceDetectorOptions', 'FaceDetectorResult', 'FaceLandmarker', 'FaceLandmarkerOptions', 'FaceLandmarkerResult', 'FaceLandmarksConnections', 'GestureRecognizer', 'GestureRecognizerOptions', 'GestureRecognizerResult', 'HandLandmarker', 'HandLandmarkerOptions', 'HandLandmarkerResult', 'HandLandmarksConnections', 'ImageClassifier', 'ImageClassifierOptions', 'ImageClassifierResult', 'ImageEmbedder', 'ImageEmbedderOptions', 'ImageEmbedderResult', 'ImageProcessingOptions', 'ImageSegmenter', 'ImageSegmenterOptions', 'InteractiveSegmenter', 'InteractiveSegmenterOptions', 'InteractiveSegmenterRegionOfInterest', 'ObjectDetector', 'ObjectDetectorOptions', 'ObjectDetectorResult', 'PoseLandmark', 'PoseLandmarker', 'PoseLandmarkerOptions', 'PoseLandmarkerResult', 'PoseLandmarksConnections', 'RunningMode', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '_

# Mudra Training Pipeline (Using GitHub Dataset)

This notebook trains a mudra recognition model using MediaPipe hand landmarks + RandomForest.

## Dataset Source
Using: https://github.com/jisharajr/Bharatanatyam-Mudra-Dataset

Expected local path after clone:
- `external/Bharatanatyam-Mudra-Dataset/`

## Steps
1. Run dataset extraction cell (creates `data/output/mudra_dataset.json`)
2. Run training cell (creates `data/output/mudra_model.pkl`)
3. Run testing cells (single-image and multi-image checks)

## Note
The extraction cell includes `MAX_IMAGES_PER_CLASS` so you can control runtime.

In [6]:
# OPTION A: If you need to extract frames from video for dataset
import cv2
import os

def extract_frames_from_video(video_path, output_folder, frame_interval=30):
    """
    Extract frames from video for creating dataset
    
    Args:
        video_path: Path to video file
        output_folder: Folder to save extracted frames
        frame_interval: Extract every Nth frame
    """
    cap = cv2.VideoCapture(video_path)
    os.makedirs(output_folder, exist_ok=True)
    
    frame_count = 0
    saved_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % frame_interval == 0:
            frame_path = os.path.join(output_folder, f"frame_{saved_count:04d}.jpg")
            cv2.imwrite(frame_path, frame)
            saved_count += 1
        
        frame_count += 1
    
    cap.release()
    print(f"Extracted {saved_count} frames from {frame_count} total frames")

# Example: Extract frames from your video
video_path = r"data\videos\sample_dance.mp4"
output_folder = r"data\extracted_frames"

# Uncomment to extract frames:
extract_frames_from_video(video_path, output_folder, frame_interval=30)

print("NOTE: For training, you need a labeled dataset with multiple mudra classes")
print("Each mudra should have its own folder with multiple images")

Extracted 20 frames from 590 total frames
NOTE: For training, you need a labeled dataset with multiple mudra classes
Each mudra should have its own folder with multiple images


In [ ]:
# Let's check what you currently have and create a practical training approach

import os
import json

# Check existing data
video_file = r"data\videos\sample_dance.mp4"
landmarks_file = r"data\output\landmarks.json"
mudras_csv = r"data\mudras.csv"

print("=" * 60)
print("CURRENT DATA STATUS")
print("=" * 60)

if os.path.exists(video_file):
    import cv2
    cap = cv2.VideoCapture(video_file)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    duration = frame_count / fps if fps > 0 else 0
    cap.release()
    print(f"✓ Video file found: {frame_count} frames, {fps} FPS, ~{duration:.1f}s duration")
else:
    print("✗ Video file not found")

if os.path.exists(landmarks_file):
    with open(landmarks_file, 'r') as f:
        landmarks_data = json.load(f)
    print(f"✓ Landmarks data: {len(landmarks_data)} frames extracted")
else:
    print("✗ No landmarks data")

if os.path.exists(mudras_csv):
    import pandas as pd
    mudras_df = pd.read_csv(mudras_csv)
    print(f"✓ Mudras reference: {len(mudras_df)} mudra classes defined")
else:
    print("✗ No mudras reference")

# Check for any image datasets
mudra_images_path = r"data\mudra_images"
if os.path.exists(mudra_images_path):
    folders = [f for f in os.listdir(mudra_images_path) if os.path.isdir(os.path.join(mudra_images_path, f))]
    if folders:
        print(f"\n✓ Found {len(folders)} mudra image folders:")
        for folder in folders:
            img_count = len([f for f in os.listdir(os.path.join(mudra_images_path, folder)) 
                           if f.endswith(('.jpg', '.png', '.jpeg'))])
            print(f"  - {folder}: {img_count} images")
    else:
        print("\n⚠ mudra_images folder exists but is empty")
else:
    print(f"\n✗ No mudra_images folder found")
    print(f"   Expected at: {mudra_images_path}")

print("\n" + "=" * 60)
print("RECOMMENDED NEXT STEPS:")
print("=" * 60)
print("""
You have 3 options:

1. CREATE DATASET FROM VIDEO (Manual labeling required):
   - Extract frames from your video
   - Manually sort frames into mudra class folders
   - Then train the model
   
2. DOWNLOAD PUBLIC DATASET:
   - Search for "Bharatanatyam mudra dataset" or "Indian classical dance mudra images"
   - Organize images into: data/mudra_images/[mudra_name]/images.jpg
   
3. USE TEMPORAL MODEL (Advanced):
   - Train on sequences from your existing landmarks.json
   - Requires temporal annotations (which frame = which mudra)

Which option would you like to proceed with?
""")

CURRENT DATA STATUS
✓ Video file found: 590 frames, 25 FPS, ~23.6s duration
✓ Landmarks data: 590 frames extracted
✓ Mudras reference: 37 mudra classes defined

✗ No mudra_images folder found
   Expected at: c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\mudra_images

RECOMMENDED NEXT STEPS:

You have 3 options:

1. CREATE DATASET FROM VIDEO (Manual labeling required):
   - Extract frames from your video
   - Manually sort frames into mudra class folders
   - Then train the model

2. DOWNLOAD PUBLIC DATASET:
   - Search for "Bharatanatyam mudra dataset" or "Indian classical dance mudra images"
   - Organize images into: data/mudra_images/[mudra_name]/images.jpg

3. USE TEMPORAL MODEL (Advanced):
   - Train on sequences from your existing landmarks.json
   - Requires temporal annotations (which frame = which mudra)

Which option would you like to proceed with?



## Quick Start: Create Demo Dataset

Since you don't have labeled mudra images yet, let's create a simple demo dataset using synthetic/sample data to test the training pipeline. Once you have real images, you can replace them.

In [ ]:
# OPTION: Extract frames from your video with hand detection visualization
# This will help you see which frames have good hand visibility

import cv2
import os
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request

# Download hand landmarker model if not present
model_path = 'hand_landmarker.task'
if not os.path.exists(model_path):
    print("Downloading MediaPipe hand model...")
    model_url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
    urllib.request.urlretrieve(model_url, model_path)
    print("✓ Model downloaded")

# Initialize hand detector
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

# Extract frames with hand detection
video_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\videos\sample_dance.mp4"
output_folder = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\extracted_frames"

os.makedirs(output_folder, exist_ok=True)

cap = cv2.VideoCapture(video_path)
frame_count = 0
saved_count = 0
hand_detected_frames = []

# Extract every 15th frame (to get ~40 frames from your video)
frame_interval = 15

print(f"Extracting frames with hand detection...")
print(f"Extracting every {frame_interval}th frame")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_count % frame_interval == 0:
        # Save frame
        frame_path = os.path.join(output_folder, f"frame_{frame_count:04d}.jpg")
        cv2.imwrite(frame_path, frame)
        
        # Check if hand is detected
        try:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            result = detector.detect(mp_image)
            
            if result.hand_landmarks and len(result.hand_landmarks) > 0:
                hand_detected_frames.append(frame_count)
                saved_count += 1
        except:
            pass
    
    frame_count += 1

cap.release()

print(f"\n✓ Extracted {saved_count} frames with hands detected")
print(f"  Total frames processed: {frame_count}")
print(f"  Frames saved to: {output_folder}")
print(f"\nFrames with hands: {hand_detected_frames[:10]}..." if len(hand_detected_frames) > 10 else f"\nFrames with hands: {hand_detected_frames}")

print("\n" + "="*60)
print("NEXT STEP: Organize these frames into mudra folders")
print("="*60)
print(f"""
1. Go to: {output_folder}
2. Create folders for each mudra (e.g., 'pataka', 'tripataka', etc.)
3. Move/copy relevant frames into each mudra folder
4. Then come back and run the training cells

Example structure:
  data/mudra_images/
    pataka/
      frame_0030.jpg
      frame_0045.jpg
    tripataka/
      frame_0060.jpg
      frame_0075.jpg
""")

In [2]:
# QUICK DEMO: Create a synthetic training dataset from your video
# This creates a sample dataset automatically so you can test the training pipeline

import cv2
import os
import shutil

# Create a demo dataset by automatically splitting video frames into 3-5 mudra classes
# (You'll need to manually label these properly later for real training)

video_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\videos\sample_dance.mp4"
demo_dataset_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\mudra_images"

# Create demo mudra folders (using first 5 mudras from your CSV)
demo_mudras = ['Pataka', 'Tripataka', 'Ardhachandra', 'Kartarimukha', 'Arala']

print("Creating demo dataset structure...")
for mudra in demo_mudras:
    mudra_folder = os.path.join(demo_dataset_path, mudra)
    os.makedirs(mudra_folder, exist_ok=True)
    print(f"✓ Created folder: {mudra}")

# Extract frames and distribute them across mudra folders
cap = cv2.VideoCapture(video_path)
frame_count = 0
saved_per_class = 0
current_mudra_idx = 0
images_per_mudra = 10  # Extract 10 images per mudra class

print(f"\nExtracting {images_per_mudra} sample images per mudra class...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Distribute frames evenly across mudra classes
    mudra_name = demo_mudras[current_mudra_idx]
    
    # Extract every 4th frame
    if frame_count % 4 == 0 and saved_per_class < images_per_mudra:
        mudra_folder = os.path.join(demo_dataset_path, mudra_name)
        frame_path = os.path.join(mudra_folder, f"{mudra_name.lower()}_{saved_per_class:03d}.jpg")
        cv2.imwrite(frame_path, frame)
        saved_per_class += 1
        
        # Move to next mudra class
        if saved_per_class >= images_per_mudra:
            saved_per_class = 0
            current_mudra_idx += 1
            if current_mudra_idx >= len(demo_mudras):
                break
    
    frame_count += 1

cap.release()

print(f"\n✓ Demo dataset created!")
print(f"  Location: {demo_dataset_path}")
print(f"  Classes: {len(demo_mudras)}")
print(f"  Images per class: ~{images_per_mudra}")

# Verify dataset
print("\nDataset summary:")
for mudra in demo_mudras:
    mudra_folder = os.path.join(demo_dataset_path, mudra)
    img_count = len([f for f in os.listdir(mudra_folder) if f.endswith('.jpg')])
    print(f"  {mudra}: {img_count} images")

print("\n" + "="*60)
print("⚠ NOTE: This is a DEMO dataset!")
print("="*60)
print("""
The frames are randomly assigned to mudra classes for testing purposes.
For REAL training, you need to:
1. Manually review and correct the labels
2. Or use properly labeled mudra images
3. Or collect your own labeled dataset

But you can proceed with training to test the pipeline!
""")

Creating demo dataset structure...
✓ Created folder: Pataka
✓ Created folder: Tripataka
✓ Created folder: Ardhachandra
✓ Created folder: Kartarimukha
✓ Created folder: Arala

Extracting 10 sample images per mudra class...

✓ Demo dataset created!
  Location: c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\mudra_images
  Classes: 5
  Images per class: ~10

Dataset summary:
  Pataka: 10 images
  Tripataka: 10 images
  Ardhachandra: 10 images
  Kartarimukha: 10 images
  Arala: 10 images

⚠ NOTE: This is a DEMO dataset!

The frames are randomly assigned to mudra classes for testing purposes.
For REAL training, you need to:
1. Manually review and correct the labels
2. Or use properly labeled mudra images
3. Or collect your own labeled dataset

But you can proceed with training to test the pipeline!



In [9]:
import os
import json
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm import tqdm
import urllib.request
from collections import Counter

# ==============================
# CONFIG
# ==============================
DATASET_PATH = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\external\Bharatanatyam-Mudra-Dataset"
OUTPUT_PATH = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\output\mudra_dataset.json"
MAX_IMAGES_PER_CLASS = 300  # set None to use all images

# ==============================
# MediaPipe model setup
# ==============================
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5
)

try:
    detector = vision.HandLandmarker.create_from_options(options)
    print("✓ MediaPipe Hand Landmarker initialized")
except Exception:
    print("⚠ Model file not found. Downloading...")
    model_url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
    urllib.request.urlretrieve(model_url, 'hand_landmarker.task')
    detector = vision.HandLandmarker.create_from_options(options)
    print("✓ Model downloaded and initialized")


def normalize_class_name(name: str) -> str:
    # Example: "Aralam(1)" -> "Aralam", "Katakamukha_1" remains
    return name.replace("(1)", "").strip()


def extract_landmarks(image_path):
    image = mp.Image.create_from_file(image_path)
    detection_result = detector.detect(image)

    if detection_result.hand_landmarks and len(detection_result.hand_landmarks) > 0:
        hand = detection_result.hand_landmarks[0]
        landmarks = []
        for lm in hand:
            landmarks.extend([lm.x, lm.y, lm.z])
        return landmarks
    return None


if not os.path.exists(DATASET_PATH):
    print(f"❌ Dataset path not found: {DATASET_PATH}")
    print("Clone the repo first into external/Bharatanatyam-Mudra-Dataset")
else:
    class_folders = [
        f for f in os.listdir(DATASET_PATH)
        if os.path.isdir(os.path.join(DATASET_PATH, f)) and not f.startswith('.')
    ]

    print(f"✓ Using dataset at: {DATASET_PATH}")
    print(f"✓ Found class folders: {len(class_folders)}")

    dataset = []
    per_class_counts = Counter()

    for cls in tqdm(sorted(class_folders), desc="Processing classes"):
        folder = os.path.join(DATASET_PATH, cls)
        class_name = normalize_class_name(cls)

        images = [
            f for f in os.listdir(folder)
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".jfif", ".webp"))
        ]

        if MAX_IMAGES_PER_CLASS is not None:
            images = images[:MAX_IMAGES_PER_CLASS]

        for img in tqdm(images, desc=class_name, leave=False):
            img_path = os.path.join(folder, img)
            try:
                landmarks = extract_landmarks(img_path)
                if landmarks:
                    dataset.append({
                        "mudra": class_name,
                        "landmarks": landmarks,
                        "image": img,
                        "source_folder": cls
                    })
                    per_class_counts[class_name] += 1
            except Exception:
                continue

    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2)

    print(f"\n✓ Dataset saved: {OUTPUT_PATH}")
    print(f"✓ Total valid samples (hand detected): {len(dataset)}")
    print("\nTop classes by valid samples:")
    for name, cnt in per_class_counts.most_common(15):
        print(f"  {name}: {cnt}")

✓ MediaPipe Hand Landmarker initialized
✓ Using dataset at: c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\external\Bharatanatyam-Mudra-Dataset
✓ Found class folders: 50


Processing classes: 100%|██████████| 50/50 [18:27<00:00, 22.14s/it] 



✓ Dataset saved: c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\output\mudra_dataset.json
✓ Total valid samples (hand detected): 11754

Top classes by valid samples:
  Kartariswastika: 298
  Kilaka: 298
  Sakata: 297
  Garuda: 296
  Anjali: 294
  Ardhachandran: 294
  Chakra: 294
  Pasha: 292
  Berunda: 288
  Kapotham: 288
  Karkatta: 287
  Katakavardhana: 283
  Varaha: 281
  Mayura: 278
  Shanka: 278


In [10]:
import json
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import pickle

# Load dataset
dataset_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\output\mudra_dataset.json"

with open(dataset_path) as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples")

X = []
y = []

for item in data:
    X.append(item["landmarks"])
    y.append(item["mudra"])

X = np.array(X)

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Classes: {label_encoder.classes_}")
print(f"Number of classes: {len(label_encoder.classes_)}")

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Train model
print("\nTraining RandomForest model...")
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✓ Model trained successfully!")
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Save model
model_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\output\mudra_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump((model, label_encoder), f)

print(f"\n✓ Model saved to: {model_path}")

Loaded 11754 samples
Classes: ['Alapadmam' 'Anjali' 'Aralam' 'Ardhachandran' 'Ardhapathaka' 'Berunda'
 'Bramaram' 'Chakra' 'Chandrakala' 'Chaturam' 'Garuda' 'Hamsapaksha'
 'Hamsasyam' 'Kangulam' 'Kapith' 'Kapotham' 'Karkatta' 'Kartariswastika'
 'Katakamukha_1' 'Katakamukha_2' 'Katakamukha_3' 'Katakavardhana'
 'Katrimukha' 'Khatva' 'Kilaka' 'Kurma' 'Matsya' 'Mayura' 'Mrigasirsha'
 'Mukulam' 'Mushti' 'Nagabandha' 'Padmakosha' 'Pasha' 'Pathaka'
 'Pushpaputa' 'Sakata' 'Samputa' 'Sarpasirsha' 'Shanka' 'Shivalinga'
 'Shukatundam' 'Sikharam' 'Simhamukham' 'Suchi' 'Tamarachudam'
 'Tripathaka' 'Trishulam' 'Varaha']
Number of classes: 49

Training samples: 9403
Testing samples: 2351

Training RandomForest model...

✓ Model trained successfully!
Test Accuracy: 0.9298 (92.98%)

Classification Report:
                 precision    recall  f1-score   support

      Alapadmam       0.88      0.90      0.89        31
         Anjali       0.96      0.92      0.94        59
         Aralam       0.89  

In [11]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pickle
import os

# Load trained model
model_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\output\mudra_model.pkl"
with open(model_path, "rb") as f:
    model, label_encoder = pickle.load(f)

print("✓ Model loaded successfully")
print(f"Classes: {len(label_encoder.classes_)}")

# Initialize MediaPipe Hand Landmarker
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)


def extract_landmarks(image_path):
    image = mp.Image.create_from_file(image_path)
    detection_result = detector.detect(image)

    if detection_result.hand_landmarks and len(detection_result.hand_landmarks) > 0:
        hand = detection_result.hand_landmarks[0]
        landmarks = []
        for lm in hand:
            landmarks.extend([lm.x, lm.y, lm.z])
        return landmarks
    return None


def predict_mudra(image_path):
    if not os.path.exists(image_path):
        print(f"Error: Image not found at {image_path}")
        return None

    landmarks = extract_landmarks(image_path)

    if landmarks is None:
        print("No hand detected in the image")
        return None

    prediction = model.predict([landmarks])
    mudra_name = label_encoder.inverse_transform(prediction)[0]

    probabilities = model.predict_proba([landmarks])[0]
    confidence = probabilities[prediction[0]]

    print(f"\n✓ Predicted Mudra: {mudra_name}")
    print(f"  Confidence: {confidence:.4f} ({confidence*100:.2f}%)")

    top_3_idx = np.argsort(probabilities)[-3:][::-1]
    print("\n  Top 3 predictions:")
    for idx in top_3_idx:
        print(f"    {label_encoder.classes_[idx]}: {probabilities[idx]*100:.2f}%")

    return mudra_name

# Test on your image in data/
test_image_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\testimage.jfif"

print(f"Testing on: {test_image_path}")
if os.path.exists(test_image_path):
    _ = predict_mudra(test_image_path)
else:
    print("⚠ Test image not found")

✓ Model loaded successfully
Classes: 49
Testing on: c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\testimage.jfif

✓ Predicted Mudra: Kapotham
  Confidence: 0.1450 (14.50%)

  Top 3 predictions:
    Kapotham: 14.50%
    Nagabandha: 13.00%
    Shanka: 12.00%


In [15]:
# Test the trained model on sample images from the GitHub dataset
import glob
import os

dataset_root = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\external\Bharatanatyam-Mudra-Dataset"

# Build a quick folder map: normalized folder name -> actual folder path
folder_map = {}
for f in os.listdir(dataset_root):
    p = os.path.join(dataset_root, f)
    if os.path.isdir(p) and not f.startswith('.'):
        normalized = f.replace("(1)", "").strip()
        folder_map[normalized] = p

# Pick up to 5 classes learned by the model
classes_to_test = list(label_encoder.classes_)[:5]
test_images = []

for cls in classes_to_test:
    cls = str(cls)
    if cls in folder_map:
        imgs = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.jfif", "*.webp"):
            imgs.extend(glob.glob(os.path.join(folder_map[cls], ext)))
        if imgs:
            test_images.append((cls, imgs[0]))

print("Testing model on sample images:\n")
print("=" * 60)

if not test_images:
    print("No test images found for model classes.")
else:
    for true_mudra, image_path in test_images:
        print(f"\nTrue label folder: {true_mudra}")
        print(f"Image: {os.path.basename(image_path)}")
        print("-" * 60)
        _ = predict_mudra(image_path)
        print("=" * 60)

print("\n✓ Multi-image test completed")

Testing model on sample images:


True label folder: Alapadmam
Image: Alapadmam_0.jpg
------------------------------------------------------------

✓ Predicted Mudra: Alapadmam
  Confidence: 0.7150 (71.50%)

  Top 3 predictions:
    Alapadmam: 71.50%
    Karkatta: 8.00%
    Ardhachandran: 4.00%

True label folder: Anjali
Image: Anjali_0.jpg
------------------------------------------------------------
No hand detected in the image

True label folder: Aralam
Image: Aralam_0.jpg
------------------------------------------------------------

✓ Predicted Mudra: Aralam
  Confidence: 0.9850 (98.50%)

  Top 3 predictions:
    Aralam: 98.50%
    Hamsasyam: 1.00%
    Shukatundam: 0.50%

True label folder: Ardhachandran
Image: Ardhachandran_0.jpg
------------------------------------------------------------

✓ Predicted Mudra: Ardhachandran
  Confidence: 0.5450 (54.50%)

  Top 3 predictions:
    Ardhachandran: 54.50%
    Chakra: 13.50%
    Mushti: 7.00%

True label folder: Ardhapathaka
Image: Ardha

## ✅ Training Pipeline Complete!

You've successfully completed the mudra recognition training pipeline:

### What Was Done:
- ✓ Created demo dataset (50 images in 5 mudra classes)
- ✓ Extracted hand landmarks (8 samples with detected hands)
- ✓ Trained RandomForest model (100% accuracy on 2 classes)
- ✓ Model saved and tested

### Limitations of Current Model:
- Only 8 training samples (very small dataset)
- Only 2 mudra classes learned (Arala, Kartarimukha)
- Frames from video may not show clear hand mudras
- Hand detection failed on some images

### To Improve the Model:

1. **Better Dataset**:
   - Get 50-100 images per mudra class
   - Images should have clear, visible hands
   - Use proper lighting and camera angles
   - Consider downloading a Bharatanatyam mudra dataset

2. **More Classes**:
   - Add all 37 mudras from your mudras.csv reference
   - Ensure balanced dataset (similar number of images per class)

3. **Better Images**:
   - Capture hands against plain backgrounds
   - Ensure hands are centered and fully visible
   - Use consistent lighting

### Files Created:
- `data/mudra_images/` - Training dataset
- `data/output/mudra_dataset.json` - Extracted landmarks
- `data/output/mudra_model.pkl` - Trained model
- `hand_landmarker.task` - MediaPipe model

### To Use This Model:
Run the prediction cell with a valid hand image path to classify mudras!

In [16]:
# Quick test with your new image
test_image_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\testimage.jfif"
print(f"Testing on: {test_image_path}")
predict_mudra(test_image_path)

Testing on: c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\testimage.jfif

✓ Predicted Mudra: Kapotham
  Confidence: 0.1450 (14.50%)

  Top 3 predictions:
    Kapotham: 14.50%
    Nagabandha: 13.00%
    Shanka: 12.00%


np.str_('Kapotham')

In [17]:
# Diagnostic: Visualize what model sees on your test image
import cv2
from mediapipe.tasks.python import vision

test_image_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\testimage.jfif"
image = cv2.imread(test_image_path)
h, w, c = image.shape

print(f"Image dimensions: {w} x {h}")

# Detect hands
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
detection_result = detector.detect(mp_image)

if detection_result.hand_landmarks and len(detection_result.hand_landmarks) > 0:
    hand = detection_result.hand_landmarks[0]
    
    # Get bounding box
    x_coords = [lm.x for lm in hand]
    y_coords = [lm.y for lm in hand]
    
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)
    
    # Convert to pixel coords with padding
    padding = 0.1
    x_min_px = max(0, int((x_min - padding) * w))
    x_max_px = min(w, int((x_max + padding) * w))
    y_min_px = max(0, int((y_min - padding) * h))
    y_max_px = min(h, int((y_max + padding) * h))
    
    print(f"\nHand detected:")
    print(f"  Bounding box: ({x_min_px}, {y_min_px}) to ({x_max_px}, {y_max_px})")
    print(f"  Hand width: {x_max_px - x_min_px}px ({(x_max - x_min)*100:.1f}% of image)")
    print(f"  Hand height: {y_max_px - y_min_px}px ({(y_max - y_min)*100:.1f}% of image)")
    
    # Crop and test
    hand_crop = image[y_min_px:y_max_px, x_min_px:x_max_px]
    crop_path = r"c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\hand_crop.jpg"
    cv2.imwrite(crop_path, hand_crop)
    
    print(f"\nCropped hand saved to: {crop_path}")
    print("\nTesting on cropped hand region:")
    _ = predict_mudra(crop_path)
    
else:
    print("❌ No hand detected in image!")

Image dimensions: 1018 x 1600

Hand detected:
  Bounding box: (595, 149) to (901, 573)
  Hand width: 306px (10.1% of image)
  Hand height: 424px (6.5% of image)

Cropped hand saved to: c:\Users\Sanjay\Downloads\minor_sem_6-main\minor_sem_6-main\data\hand_crop.jpg

Testing on cropped hand region:

✓ Predicted Mudra: Mushti
  Confidence: 0.1550 (15.50%)

  Top 3 predictions:
    Mushti: 15.50%
    Chakra: 12.00%
    Samputa: 8.50%
